# The ML Lab by hand
Four experiments from the dashboard's ML Lab in plain pandas, scikit-learn and LightGBM. Data: `demo_bundle/lab/` (real rows of the smartci study).

In [1]:
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score as pr_auc
from pathlib import Path

lab = next(p / "demo_bundle" / "lab" for p in (Path.cwd(), Path.cwd().parent) if (p / "demo_bundle").exists())
train = pd.read_parquet(lab / "train_sample.parquet")   # training rows (failures over-sampled)
later = pd.read_parquet(lab / "later_builds.parquet")   # CI runs AFTER training, real failure rate
history = [c for c in train.columns if c.startswith("hs_")] + ["is_new_test"]
change = ["test_file_in_diff", "same_top_module", "path_prefix_overlap", "name_token_overlap",
          "bm25_test_vs_diff", "chg_n_files", "chg_additions", "chg_deletions", "chg_n_top_modules",
          "has_change_data"]
cost = ["dur_last_asof", "dur_ema_asof", "dur_source"]
cofail = ["cof_max", "cof_sum", "cof_count_max"]
features = history + cost + change + cofail
print(len(train), "training rows,", len(later), "later rows; failure rate later:", later.y_fail.mean())

100000 training rows, 100189 later rows; failure rate later: 0.0017566798750361817


## 1. The accuracy trap

In [2]:
X = StandardScaler().fit(train[features])                      # put features on one scale
lr = LogisticRegression(max_iter=1000).fit(X.transform(train[features]), train.y_fail)
p = lr.predict_proba(X.transform(later[features]))[:, 1]
print("model accuracy        ", ((p >= 0.5) == later.y_fail).mean())
print("always say 'pass'     ", 1 - later.y_fail.mean())      # a useless model scores this
print("model PR-AUC          ", pr_auc(later.y_fail, p), " random:", later.y_fail.mean())
# Takeaway: with 0.2% failures accuracy rewards doing nothing; PR-AUC measures the ranking.

model accuracy         0.9954386210062981
always say 'pass'      0.9982433201249639
model PR-AUC           0.21963200998756938  random: 0.0017566798750361817


## 2. Overfitting without regularization

In [3]:
def fit_lgbm(df, leaves, min_leaf, rounds, valid=None):
    params = dict(objective="binary", num_leaves=leaves, min_child_samples=min_leaf, verbose=-1)
    data = lgb.Dataset(df[features], df.y_fail)
    if valid is None:                                          # no early stopping
        return lgb.train(params, data, rounds)
    v = lgb.Dataset(valid[features], valid.y_fail, reference=data)
    return lgb.train(params, data, rounds, valid_sets=[v], callbacks=[lgb.early_stopping(30, verbose=False)])

runs = train[["project", "run_idx", "start_ts"]].drop_duplicates().sort_values("start_ts")
late = runs.iloc[int(len(runs) * 0.8):]                        # latest 20% of CI runs = validation
is_val = train.set_index(["project", "run_idx"]).index.isin(late.set_index(["project", "run_idx"]).index)
fit, val = train[~is_val], train[is_val]

for name, model in [("no brakes", fit_lgbm(fit, 127, 1, 200)),
                    ("regularized", fit_lgbm(fit, 31, 50, 300, valid=val))]:
    print(name, "train", round(pr_auc(fit.y_fail, model.predict(fit[features])), 3),
          "later", round(pr_auc(later.y_fail, model.predict(later[features])), 3))
# Takeaway: a perfect training score is a warning sign; judge a model on data it has never seen.

no brakes train 1.0 later 0.063


regularized train 0.637 later 0.165


## 3. A random split inflates the score

In [4]:
rand_val = np.random.default_rng(0).random(len(train)) < 0.2       # random 20% of rows
for name, mask in [("random split", rand_val), ("split by time", is_val)]:
    model = fit_lgbm(train[~mask], 127, 1, 200)
    print(name, "validation", round(pr_auc(train[mask].y_fail, model.predict(train[mask][features])), 3),
          "later builds", round(pr_auc(later.y_fail, model.predict(later[features])), 3))
# Takeaway: rows of one CI run are similar, so a random split leaks; split by time like real use.

random split validation 0.45 later builds 0.087


split by time validation 0.241 later builds 0.063


## 4. History alone vs history + change

In [5]:
for name, cols in [("history only", history), ("history + cost + change", history + cost + change)]:
    data, v = lgb.Dataset(fit[cols], fit.y_fail), lgb.Dataset(val[cols], val.y_fail)
    model = lgb.train(dict(objective="binary", verbose=-1), data, 300, valid_sets=[v],
                      callbacks=[lgb.early_stopping(30, verbose=False)])
    p = model.predict(later[cols])
    print(name, "later PR-AUC", round(pr_auc(later.y_fail, p), 3),
          "first-time failures", round(pr_auc(later.y_ftf_asof, p), 4), "random", later.y_ftf_asof.mean())
# Takeaway: history finds repeat failures; tests failing for the first time have no history to use.

history only later PR-AUC 0.18 first-time failures 0.0002 random 8.983022088253202e-05


history + cost + change later PR-AUC 0.084 first-time failures 0.0003 random 8.983022088253202e-05
